   
   
## Migrate Schema: `hls_glucosphere.cgm` → `hls_amer_catalog.glucosphere`

**Author:** May Merkletan  
**Date:** 2026-03-12  
**Purpose:** Bulk-migrate all **tables**, **volumes**, and **models** from `hls_glucosphere.cgm` into `hls_amer_catalog.glucosphere`. A `cgm_` prefix on table names is **optional** (controlled by the `apply_prefix` toggle). Volumes and models always keep their original names.

### Task Plan
| Step | Task | Details |
| --- | --- | --- |
| 1 | **Setup & Widgets** | Define source/destination paths, prefix value, and apply-prefix toggle |
| 2 | **List Source Tables** | Dynamically query `SHOW TABLES` to collect all 34 table names |
| 3 | **Copy Tables** | Drop existing destination tables, then `CREATE TABLE … AS SELECT *` (with optional prefix) |
| 4 | **Verify Tables** | Compare table counts and row counts between source and destination |
| 5 | **Copy Volumes** | Drop any incorrectly prefixed volumes, then create and copy files (no prefix) |
| 6 | **Re-register Models** | Drop any incorrectly prefixed models, then register in destination (no prefix) |
| 7 | **Final Verification** | Confirm volumes, models, and tables all exist in destination |

### Parameters
| Parameter | Type | Default | Description |
| --- | --- | --- | --- |
| `source` | text | `hls_glucosphere.cgm` | Source catalog.schema |
| `destination` | text | `hls_amer_catalog.glucosphere` | Destination catalog.schema |
| `prefix` | text | `cgm_` | Prefix value for table names |
| `apply_prefix` | dropdown | `No` | Whether to apply the prefix (`Yes` / `No`) |

### Asset Inventory
| Asset Type | Count | Prefix | Items |
| --- | --- | --- | --- |
| Tables | 34 | optional `cgm_` | `baseline_for_validation_7d`, `diabetes_data`, `gold_patient_device_readings`, … |
| Volumes | 4 | _(none)_ | `data`, `diabetes_who_pdf`, `landing_zone`, `optuna_studies` |
| Models | 2 | _(none)_ | `cgm_xgb_30m` (v1), `cgm_xgb_15m` (v1) |

In [0]:
dbutils.widgets.removeAll()

In [0]:
# -- Read parameter values ---------------------------------------------------
try:
    source: str = dbutils.widgets.get("source")
except Exception:
    source = "hls_glucosphere.cgm"

try:
    destination: str = dbutils.widgets.get("destination")
except Exception:
    destination = "hls_amer_catalog.glucosphere"

try:
    prefix_value: str = dbutils.widgets.get("prefix")
except Exception:
    prefix_value = "cgm_"

try:
    apply_prefix: str = dbutils.widgets.get("apply_prefix")
except Exception:
    apply_prefix = "No"

# Effective prefix: use the value when toggle is "Yes", otherwise empty
prefix: str = prefix_value if apply_prefix == "Yes" else ""

# -- Derived config ---------------------------------------------------------
src_catalog, src_schema   = source.split(".")
dest_catalog, dest_schema = destination.split(".")

print(f"Source       : {src_catalog}.{src_schema}")
print(f"Destination  : {dest_catalog}.{dest_schema}")
print(f"Prefix value : {repr(prefix_value)}")
print(f"Apply prefix : {apply_prefix}")
print(f"Effective    : {repr(prefix) if prefix else '(none — tables keep original names)'}")

### Step 2 — List Source Tables

In [0]:
# Collect all table names from the source schema
tables_df = spark.sql(f"SHOW TABLES IN {src_catalog}.{src_schema}")
table_names: list[str] = [row.tableName for row in tables_df.collect()]

print(f"Found {len(table_names)} tables in {source}:")
for t in table_names:
    print(f"  {t}  →  {dest_catalog}.{dest_schema}.{prefix}{t}")

In [0]:
# # Find all tables with cgm_ prefix in destination schema
# dest_all_tables = [
#     row.tableName
#     for row in spark.sql(f"SHOW TABLES IN hls_amer_catalog.glucosphere").collect()
#     if row.tableName.startswith("cgm_")
# ]
# print(f"Found {len(dest_all_tables)} tables with 'cgm_' prefix to drop:\n")

# for tbl in dest_all_tables:
#     fqn = f"hls_amer_catalog.glucosphere.{tbl}"
#     spark.sql(f"DROP TABLE IF EXISTS {fqn}")
#     print(f"  🗑  Dropped {fqn}")

# print(f"\n--- Done: {len(dest_all_tables)} tables dropped ---")

### Step 3 — Copy Tables with Prefix

In [0]:
from pyspark.sql.utils import AnalysisException

success: list[str] = []
failures: list[tuple[str, str]] = []

for tbl in table_names:
    src_fqn  = f"{src_catalog}.{src_schema}.{tbl}"
    dest_fqn = f"{dest_catalog}.{dest_schema}.{prefix}{tbl}"
    try:
        spark.sql(f"DROP TABLE IF EXISTS {dest_fqn}")
        spark.sql(f"CREATE TABLE {dest_fqn} AS SELECT * FROM {src_fqn}")
        success.append(tbl)
        print(f"  ✓  {dest_fqn}")
    except AnalysisException as e:
        failures.append((tbl, str(e)))
        print(f"  ✗  {dest_fqn}  — {e}")

print(f"\n--- Done: {len(success)} succeeded, {len(failures)} failed ---")

### Step 4 — Verify Copied Tables

In [0]:
import pyspark.sql.functions as F

# List destination tables that came from our copy
# When prefix is set, filter by prefix; otherwise match against source table names
src_table_set = set(table_names)
if prefix:
    dest_table_names: list[str] = [
        row.tableName
        for row in spark.sql(f"SHOW TABLES IN {dest_catalog}.{dest_schema}").collect()
        if row.tableName.startswith(prefix)
    ]
else:
    dest_table_names: list[str] = [
        row.tableName
        for row in spark.sql(f"SHOW TABLES IN {dest_catalog}.{dest_schema}").collect()
        if row.tableName in src_table_set
    ]

print(f"Destination tables{f' with prefix {repr(prefix)}' if prefix else ''}: {len(dest_table_names)} / {len(table_names)} expected")

# Row-count comparison
rows: list[dict] = []
for tbl in table_names:
    src_fqn  = f"{src_catalog}.{src_schema}.{tbl}"
    dest_fqn = f"{dest_catalog}.{dest_schema}.{prefix}{tbl}"
    src_cnt  = spark.sql(f"SELECT COUNT(*) AS cnt FROM {src_fqn}").first().cnt
    try:
        dest_cnt = spark.sql(f"SELECT COUNT(*) AS cnt FROM {dest_fqn}").first().cnt
    except Exception:
        dest_cnt = -1
    rows.append({"table": tbl, "source_rows": src_cnt, "dest_rows": dest_cnt, "match": src_cnt == dest_cnt})

verify_df = spark.createDataFrame(rows)
display(verify_df.orderBy("table"))

   
### Step 5 — Copy Volumes (no prefix)
Drop any previously created `cgm_`-prefixed volumes, then create volumes in the destination schema **using the original names** and recursively copy files.

**Source volumes (4):** `data`, `diabetes_who_pdf`, `landing_zone`, `optuna_studies`

In [0]:
# --- Drop incorrectly prefixed volumes first ---
dest_existing_vols = [
    row.volume_name
    for row in spark.sql(f"SHOW VOLUMES IN {dest_catalog}.{dest_schema}").collect()
    if row.volume_name.startswith(prefix)
]
for bad_vol in dest_existing_vols:
    spark.sql(f"DROP VOLUME IF EXISTS {dest_catalog}.{dest_schema}.{bad_vol}")
    print(f"  🗑  Dropped prefixed volume: {dest_catalog}.{dest_schema}.{bad_vol}")

# --- List source volumes dynamically ---
vol_names: list[str] = [
    row.volume_name
    for row in spark.sql(f"SHOW VOLUMES IN {src_catalog}.{src_schema}").collect()
]
print(f"\nFound {len(vol_names)} volumes in {source}")

vol_success: list[str] = []
vol_failures: list[tuple[str, str]] = []

for vol in vol_names:
    # No prefix for volumes — keep original name
    src_path  = f"/Volumes/{src_catalog}/{src_schema}/{vol}"
    dest_path = f"/Volumes/{dest_catalog}/{dest_schema}/{vol}"
    try:
        # Create managed volume in destination (idempotent)
        spark.sql(f"CREATE VOLUME IF NOT EXISTS {dest_catalog}.{dest_schema}.{vol}")
        # Recursively copy files
        dbutils.fs.cp(src_path, dest_path, recurse=True)
        vol_success.append(vol)
        print(f"  ✓  {dest_path}")
    except Exception as e:
        vol_failures.append((vol, str(e)))
        print(f"  ✗  {dest_path}  — {e}")

print(f"\n--- Volumes: {len(vol_success)} succeeded, {len(vol_failures)} failed ---")

   
### Step 6 — Re-register Models (no prefix)
Drop any previously created `cgm_`-prefixed model copies, then copy model versions from `hls_glucosphere.cgm` to `hls_amer_catalog.glucosphere` **using the original names**.

**Source models (2):** `cgm_xgb_30m` (v1), `cgm_xgb_15m` (v1)

In [0]:
import mlflow
from mlflow import MlflowClient

mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

# --- Drop incorrectly prefixed models first ---
all_dest_models_existing = client.search_registered_models(max_results=200)
bad_models = [
    m for m in all_dest_models_existing
    if m.name.startswith(f"{dest_catalog}.{dest_schema}.{prefix}{prefix.rstrip('_')}")
    or (m.name.startswith(f"{dest_catalog}.{dest_schema}.{prefix}")
        and m.name.split('.')[-1] not in [
            row.tableName for row in spark.sql(f"SHOW TABLES IN {dest_catalog}.{dest_schema}").collect()
        ])
]
# More targeted: drop models whose short name starts with the double-prefix pattern
for bm in all_dest_models_existing:
    short = bm.name.split('.')[-1]
    if bm.name.startswith(f"{dest_catalog}.{dest_schema}.") and short.startswith(f"{prefix}"):
        # Check if this is a prefixed copy of a source model
        candidate_src = short[len(prefix):]  # remove prefix
        src_model_names = [m.name.split('.')[-1] for m in client.search_registered_models(max_results=200) if m.name.startswith(f"{src_catalog}.{src_schema}.")]
        if candidate_src in src_model_names:
            try:
                # Delete all versions first
                for v in client.search_model_versions(f"name='{bm.name}'"):
                    client.delete_model_version(bm.name, v.version)
                client.delete_registered_model(bm.name)
                print(f"  🗑  Dropped prefixed model: {bm.name}")
            except Exception as e:
                print(f"  ⚠  Could not drop {bm.name}: {e}")

# --- Discover source models ---
all_models = client.search_registered_models(max_results=200)
src_models = [m for m in all_models if m.name.startswith(f"{src_catalog}.{src_schema}.")]
print(f"\nFound {len(src_models)} models in {source}")

model_success: list[str] = []
model_failures: list[tuple[str, str]] = []

for m in src_models:
    # Keep original name — no prefix for models
    short_name = m.name.split(".")[-1]
    dest_model_name = f"{dest_catalog}.{dest_schema}.{short_name}"

    try:
        # Ensure destination registered model exists
        try:
            client.create_registered_model(dest_model_name)
            print(f"  Created model: {dest_model_name}")
        except Exception:
            print(f"  Model already exists: {dest_model_name}")

        # Copy each version
        src_versions = client.search_model_versions(f"name='{m.name}'")
        for v in src_versions:
            copied = client.copy_model_version(
                src_model_uri=f"models:/{m.name}/{v.version}",
                dst_name=dest_model_name,
            )
            print(f"    ✓  v{v.version} → {dest_model_name} v{copied.version}")

        model_success.append(short_name)
    except Exception as e:
        model_failures.append((short_name, str(e)))
        print(f"    ✗  {dest_model_name}  — {e}")

print(f"\n--- Models: {len(model_success)} succeeded, {len(model_failures)} failed ---")

### Step 7 — Final Verification
Confirm all assets (tables, volumes, models) exist in the destination schema.

In [0]:
import mlflow
from mlflow import MlflowClient

# Ensure dependencies are available even if upstream cells were skipped
if 'vol_names' not in dir():
    vol_names = [row.volume_name for row in spark.sql(f"SHOW VOLUMES IN {src_catalog}.{src_schema}").collect()]
if 'client' not in dir():
    mlflow.set_registry_uri("databricks-uc")
    client = MlflowClient()
if 'src_models' not in dir():
    all_models = client.search_registered_models(max_results=200)
    src_models = [m for m in all_models if m.name.startswith(f"{src_catalog}.{src_schema}.")]

# --- Tables (with optional prefix) ---
src_table_set = set(table_names)
if prefix:
    dest_tbls = [
        row.tableName
        for row in spark.sql(f"SHOW TABLES IN {dest_catalog}.{dest_schema}").collect()
        if row.tableName.startswith(prefix)
    ]
else:
    dest_tbls = [
        row.tableName
        for row in spark.sql(f"SHOW TABLES IN {dest_catalog}.{dest_schema}").collect()
        if row.tableName in src_table_set
    ]

# --- Volumes (original names, no prefix) ---
src_vol_names_set = set(vol_names)
dest_vols = [
    row.volume_name
    for row in spark.sql(f"SHOW VOLUMES IN {dest_catalog}.{dest_schema}").collect()
    if row.volume_name in src_vol_names_set
]

# --- Models (original names, no prefix) ---
src_model_short_names = {m.name.split('.')[-1] for m in src_models}
all_dest_models = client.search_registered_models(max_results=200)
dest_mdls = [
    m.name for m in all_dest_models
    if m.name.startswith(f"{dest_catalog}.{dest_schema}.")
    and m.name.split('.')[-1] in src_model_short_names
]

# Summary
summary = [
    {"asset_type": "Tables",  "source_count": len(table_names), "dest_count": len(dest_tbls),  "status": "✓" if len(dest_tbls)  >= len(table_names) else "⚠"},
    {"asset_type": "Volumes", "source_count": len(vol_names),   "dest_count": len(dest_vols),  "status": "✓" if len(dest_vols)  >= len(vol_names)   else "⚠"},
    {"asset_type": "Models",  "source_count": len(src_models),  "dest_count": len(dest_mdls),  "status": "✓" if len(dest_mdls)  >= len(src_models)  else "⚠"},
]

summary_df = spark.createDataFrame(summary)
display(summary_df)